In [ ]:
/**
 * CQRS Pattern: Command Query Responsibility Segregation
*/

In [ ]:
/**
 * Problem:
 * > Humare chote mote systems mai jaha pe we have ~1lakh-1M users ho, waha pe CQRS nhi
 *   chahiye. Systems like Amazon E-Commerec m humein CQRS chahiye jo global level k
 *   upar kaam krte hai which are CRUD heavy applications.
 * > We can built Amazon like E-Commerce App in 15 days, jisme humare ek seller portal
 *   hoga jaha pe mere paas authenticated secure APIs hongi to list some products in DB,
 *   aur jab humne listing kar di to we will make a consumer facing application jisme
 *   features honge:
 *   - Order Tracking
 *   - Payment Gateway
 *   - Product Listing
 *   - Selling
 *   - Cart
 *   - Review
 *   - Returns
 *   - Whislist
 *   - Inventory
 *   - Search
 * > Isme se koi v postprocessing & preprocessing pipeline nhi hai, realtime like 
 *   websocket, video pipeline nhi hai, koi redundancy nhi hai. Ye simple bahut bada
 *   CRUD Application hai, jiske wajah se traffic bahut jyda hai jo kuch na kuch har'
 *   second kr rha hai whether it is read, write, update, delete.
 * > Amazon k logs ko agar hum dekhe to 99% GET, PUT, POST operation hi milega, but the
 *   only problem is ye teeno hi millions or even billions m hoti hogi every second.
 *   So, Amazon jaise app banane k liye humein bas time chahiye aur bada database, 
 *   ye engineering problem hai hi nahi. Aur jab hum aise problem m faste hai then
 *   database becomes the bottleneck, even we know sharding, replication, etc, still 
 *   too many repeated API calls makes our database hot.
*/

In [ ]:
/**
 * Solution: CQRS
 * > Database k andar 4 chije hoti hai, but in reality 2 chije hoti hai. Either hum
 *   DB m kuch mutation/change kr rhe honge ya fir hm read kr rhe honge. Aur kahi na
 *   kahi Amazon k andar inn dono ka load ~50%-50% k karib hoga, because order krna v
 *   ek tarah ka mutation and order dekhna v ek tarah ka read hai.
 * > Agar hum inn dono ko split kr de to kya hm apne database ka ek bada problem solve kr
 *   skte hai? Yes! We can solve ~50% problem by splitting the load.
 * 
 * > Database Operations ko hm 2 parts m divide krte hai:
 *   a. Data Mutation (Create/Update/Delete)
 *   b. Data Query    (Read)
 * > Data Mutation is an umbrella term, isliye mutate ho humne 'COMMAND' bola.
 *   So, the CQRS says 'Can we segregate data based on Command (Write) and Query (Read)
 *   Operations into separate data models?'
 * > It basically means "Data ko padhne ka model alag, data ko likhne ka model alag."
 *   - Commands update the data
 *   - Queries retrieve the data
*/

In [ ]:
/**
 * THE PROBLEM — Single Database Bottleneck:
 * 
 * Ek Database Pe Sab Kuch — Kyun Problem Hoti Hai
 *   - Chhote systems (1 lakh users tak) ke liye single DB kaafi hai.
 *   - Lekin Amazon jaise global systems mein — billions of operations per second —
 *     ek database bottleneck ban jaata hai chahe sharding/replication karo ya nahi.
 * 
 * SCALE CONTEXT:
 *   ┌──────────────────────────────────────────────────────────────────┐
 *   │  Users: 1 lakh       → Single DB kaafi hai (CQRS zaroorat nahi)  │
 *   │  Users: 10 lakh      → Sharding + Replication kaafi hai          │
 *   │  Users: 1 crore+     → CQRS sochna chahiye                       │
 *   │  Users: Amazon scale → CQRS ZAROOR chahiye                       │
 *   └──────────────────────────────────────────────────────────────────┘
 * 
 * AMAZON LIKE E-COMMERCE — FEATURES:
 *
 *   ┌───────────────────────────────────────────────────────────────────┐
 *   │  READ Heavy Features       │  WRITE Heavy Features                │
 *   ├───────────────────────────────────────────────────────────────────┤
 *   │  Product Listing / Browse  │  Place Order                         │
 *   │  Product Search            │  Update Inventory                    │
 *   │  Order Tracking            │  Add to Cart                         │
 *   │  Review Display            │  Submit Review                       │
 *   │  Wishlist View             │  Add to Wishlist                     │
 *   │  Cart View                 │  Process Payment                     │
 *   │  Recommendations           │  Return / Refund Request             │
 *   │  Inventory Check (UI)      │  Seller Product Listing              │
 *   └───────────────────────────────────────────────────────────────────┘
 * 
 * WHAT HAPPENS ON SINGLE DATABASE (The Problem):
 *
 *   Every second on Amazon:
 *   → 1M+ users browsing products      → SELECT * FROM products WHERE ...
 *   → 500K users checking order status → SELECT * FROM orders WHERE user_id=...
 *   → 200K users placing orders        → INSERT INTO orders ...
 *   → 100K users updating cart         → UPDATE cart SET ...
 *   → 50K inventory updates            → UPDATE inventory SET stock=stock-1 ...
 * 
 *   ┌──────────────────────────────────────────────────────────────────┐
 *   │                    SINGLE DATABASE                               │
 *   │                                                                  │
 *   │  ←── 1M reads/sec ─────────────────────────────────────────      │
 *   │  ←── 350K writes/sec ──────────────────────────────────────      │
 *   │                                                                  │
 *   │  ALL competing for:                                              │
 *   │  → Same table locks                                              │
 *   │  → Same connection pool                                          │
 *   │  → Same CPU and RAM                                              │
 *   │  → Same disk I/O                                                 │
 *   │                                                                  │
 *   │  RESULT: Database becomes HOT → slow → timeouts → crashes        │
 *   └──────────────────────────────────────────────────────────────────┘
 * 
 * THE CORE INSIGHT:
 * - Amazon ke logs dekhein toh 99% GET, PUT, POST operations hi hongi.
 * - Problem ye nahi ki operations complex hain.
 * - Problem ye hai ki ye MILLIONS mein hoti hain EVERY SECOND.
 * - Engineering problem nahi hai — SCALE problem hai.
 * - Aur scale problem ka solution hai: SPLIT THE LOAD.
*/

In [ ]:
/**
 * TRADITIONAL ARCHITECTURE (Before CQRS - Sab Ek Jagah):
 * > Traditional CRUD architecture mein ek hi model hai.
 * > Wahi model reads ke liye use hota hai, wahi writes ke liye.
 * > Chhote systems ke liye perfect — bade systems ke liye nightmare.
 * 
 * TRADITIONAL ARCHITECTURE DIAGRAM:
 *
 *   ┌────────────────────────────────────────────────────────────────────┐
 *   │                    CLIENT (Browser / App)                          │
 *   └────────────────────────────────────────────────────────────────────┘
 *                              │
 *                              ▼
 *   ┌────────────────────────────────────────────────────────────────────┐
 *   │                      API SERVER                                    │
 *   │                                                                    │
 *   │   GET  /products      → ProductController.list()                   │
 *   │   GET  /orders/:id    → OrderController.get()                      │
 *   │   POST /orders        → OrderController.create()                   │
 *   │   PUT  /cart          → CartController.update()                    │
 *   │   POST /reviews       → ReviewController.create()                  │
 *   │                                                                    │
 *   │   ALL going through ONE service layer                              │
 *   └────────────────────────────────────────────────────────────────────┘
 *                              │
 *                              ▼
 *   ┌────────────────────────────────────────────────────────────────────┐
 *   │                  SINGLE DATABASE MODEL                             │
 *   │                                                                    │
 *   │   class OrderRepository {                                          │
 *   │       findById(id)           ← READ                                │
 *   │       findByUser(userId)     ← READ                                │
 *   │       findAll(filters)       ← READ (complex joins!)               │
 *   │       create(orderData)      ← WRITE                               │
 *   │       updateStatus(id, st)   ← WRITE                               │
 *   │       delete(id)             ← WRITE                               │
 *   │   }                                                                │
 *   └────────────────────────────────────────────────────────────────────┘
 *                              │
 *                              ▼
 *   ┌────────────────────────────────────────────────────────────────────┐
 *   │                 SINGLE DATABASE (PostgreSQL/MySQL)                 │
 *   │                                                                    │
 *   │   orders table, products table, users table, inventory table ...   │
 *   │   SAME tables handle ALL reads AND writes                          │
 *   └────────────────────────────────────────────────────────────────────┘
 * 
 * 
 * WHAT GOES WRONG AT SCALE?
 * 
 * Input: You are scaling a system where:
 *  - Thousands of users are reading data (SELECT)
 *  - Some users are updating data (UPDATE / INSERT)
 *
 * Output: Understand why things break at scale
 * 
 *   Problem 1: READ vs WRITE LOCK (Traffic Jam Situation)
 *   +──────────────────────────────────────────────────────────────────────────────+
 *   |  Imagine:                                                                    |
 *   |  - A road where only one vehicle can pass at a time                          |
 *   |  - A truck (WRITE operation) stops in the middle                             |
 *   |                                                                              |
 *   |  What happens?                                                               |
 *   |  - All cars behind (READ requests) must wait                                 |
 *   |                                                                              |
 *   |  In database:                                                                |
 *   |  - When a WRITE happens → DB locks that row/table                            |
 *   |  - During this time → READ queries may wait                                  |
 *   |                                                                              |
 *   |  Result:                                                                     |
 *   |  - Slow response                                                             |
 *   |  - App feels stuck                                                           |
 *   |  - High latency under load                                                   |
 *   |                                                                              |
 *   |  Example:                                                                    |
 *   |  ┌───────────────────────────────────────────────────────────────────┐       |
 *   |  │  Write transaction: UPDATE orders SET status='SHIPPED' WHERE id=1 │       |
 *   |  │  → Locks the row                                                  │       |
 *   |  │  → 10,000 concurrent reads on the same table → ALL WAITING!       │       |
 *   |  │  → Even with read replicas, the write DB is still a hot spot      │       |
 *   |  └───────────────────────────────────────────────────────────────────┘       |
 *   +──────────────────────────────────────────────────────────────────────────────+
 *
 *   Problem 2: DATABASE DESIGN GOOD FOR WRITE, BAD FOR READ
 *   +──────────────────────────────────────────────────────────────────────────────+
 *   |  Database is designed like this (clean & organized):                         |
 *   |  - orders table                                                              |
 *   |  - order_items table                                                         |
 *   |  - products table                                                            |
 *   |  - users table                                                               |
 *   |  - addresses table                                                           |
 *   |  This is called: NORMALIZATION (good for writing data)                       |
 *   |                                                                              |
 *   |  But when user asks: "Show my orders"                                        |
 *   |                                                                              |
 *   |  Database has to:                                                            |
 *   |  - Go to orders                                                              |
 *   |  - Then order_items                                                          |
 *   |  - Then products                                                             |
 *   |  - Then users                                                                |
 *   |  - Then addresses                                                            |
 *   |  This is called: JOIN (combining tables)                                     |
 *   |                                                                              |
 *   |  Problem:                                                                    |
 *   |  - Too many joins                                                            |
 *   |  - Query becomes heavy                                                       |
 *   |  - Slow performance on large data                                            |
 *   |                                                                              |
 *   |  Example:                                                                    |
 *   |  ┌───────────────────────────────────────────────────────────────────┐       |
 *   |  │  Write schema (normalized):                                       │       |
 *   |  │    orders table + order_items table + products table +            │       |
 *   |  │    users table + addresses table                                  │       |
 *   |  │                                                                   │       |
 *   |  │  To show "My Orders" page (READ):                                 │       |
 *   |  │    SELECT o.*, oi.*, p.name, u.name, a.street                     │       |
 *   |  │    FROM orders o                                                  │       |
 *   |  │    JOIN order_items oi ON o.id = oi.order_id                      │       |
 *   |  │    JOIN products p ON oi.product_id = p.id                        │       |
 *   |  │    JOIN users u ON o.user_id = u.id                               │       |
 *   |  │    JOIN addresses a ON o.address_id = a.id                        │       |
 *   |  │    WHERE o.user_id = ?                                            │       |
 *   |  │                                                                   │       |
 *   |  │  5-table JOIN on millions of rows every time someone views orders!│       |
 *   |  │  Slow, expensive, unscalable.                                     │       |
 *   |  └───────────────────────────────────────────────────────────────────┘       |
 *   +──────────────────────────────────────────────────────────────────────────────+
 * 
 *   Problem 3: READ vs WRITE SCALING DIFFERENCE
 *   +──────────────────────────────────────────────────────────────────────────────+
 *   |  READ (SELECT):                                                              |
 *   |  - Happens a lot (thousands of users)                                        |
 *   |  - Can be scaled easily (replicas, caching)                                  |
 *   |                                                                              |
 *   |  WRITE (INSERT/UPDATE):                                                      |
 *   |  - Less frequent but complex                                                 |
 *   |  - Needs strong consistency                                                  |
 *   |  - Hard to scale                                                             |
 *   |                                                                              |
 *   |  Problem:                                                                    |
 *   |  - Single DB treats both same                                                |
 *   |                                                                              |
 *   |  So:                                                                         |
 *   |  - You scale everything together                                             |
 *   |  - Waste resources                                                           |
 *   |  - System becomes inefficient                                                |
 *   |                                                                              |
 *   |  Example:                                                                    |
 *   |  ┌──────────────────────────────────────────────────────────────────┐        |
 *   |  │  Reads need : more replicas, caching, horizontal scale           │        |
 *   |  │  Writes need: more compute, vertical scale, strong consistency   │        |
 *   |  │                                                                  │        |
 *   |  │  With one DB: you scale BOTH the same way → waste + inefficiency │        |
 *   |  └──────────────────────────────────────────────────────────────────┘        |
 *   +──────────────────────────────────────────────────────────────────────────────+
 * 
 * FINAL SIMPLE SUMMARY:
 * 1. One WRITE can block many READS  → traffic jam
 * 2. Clean schema helps WRITE but slows READ → too many joins
 * 3. READ and WRITE need different scaling → but DB doesn't separate them 
*/

In [ ]:
/**
 * WHY READ AND WRITE ARE FUNDAMENTALLY DIFFERENT?
 * 
 * > READ aur WRITE ka kaam alag hai, isliye unki needs bhi alag hoti hain
 * > Ek hi database ko dono ke liye perfect banana: Almost impossible at large scale
 * > Isi problem ko solve karta hai: CQRS
 * 
 *   ┌───────────────────┬────────────────────────────────┬─────────────────────────────────┐
 *   │ Aspect            │ WRITE (Command)                │ READ (Query)                    │
 *   ├───────────────────┼────────────────────────────────┼─────────────────────────────────┤
 *   │ Primary need      │ - Data correct hona chahiye    │ - Fast response chahiye         │
 *   │ Consistency       │ - Strong (exact data)          │ - Thoda stale chalega           │
 *   │ Schema style      │ - Normalized (separate tables) │ - Denormalized (ready data)     │
 *   │ Complexity        │ - Validation heavy             │ - Filtering/sorting heavy       │
 *   │ Scaling           │ - Vertical (strong machine)    │ - Horizontal (multiple replicas)│
 *   │ Database type     │ - SQL (MySQL/PostgreSQL)       │ - NoSQL / Cache / Search        │
 *   │ Transactions      │ - Required                     │ - Rarely needed                 │
 *   │ Joins             │ - Allowed                      │ - Avoid (slow ho jata hai)      │
 *   │ Latency tolerance │ - 50–100ms acceptable          │ - <10ms expected                │
 *   │ Failure handling  │ - Rollback (undo changes)      │ - Old data dikha sakte ho       │
 *   └───────────────────┴────────────────────────────────┴─────────────────────────────────┘
 *   
 * 
 * WRITE SIDE NEEDS:
 *   → ACID transactions  (order fail ho to sab rollback ho jaye)
 *   → Validation         (stock available hai? payment valid hai?)
 *   → Normalized schema  (duplicate data avoid karo)
 *   → Audit trail        (kya change hua, kisne kiya, kab kiya)
 *   → Strong consistency (har jagah same data)
 *
 * READ SIDE NEEDS:
 *   → Speed                                      (user 200ms se zyada wait nahi karna chahta)
 *   → Denormalized data                          (ek query mein sab data — no JOINs)
 *   → Pagination, filtering, sorting efficiently (fast browsing)
 *   → Can tolerate slightly stale data           (2-3 second delay OK)
 *   → Horizontal scaling                         (multiple read servers)
 *   → Search capabilities                        (advanced filtering)
 *   → Caching friendly                           (Redis mein store kar sakte ho)
 *
 * Different Databases for Different Needs:
 * > WRITE  → PostgreSQL          (ACID, normalized, strong consistency)
 * > READ   → MongoDB / DynamoDB  (denormalized, fast, scalable)
 * > SEARCH → Elasticsearch       (full-text search, filters)
 * > CACHE  → Redis               (sub-millisecond reads)
 *
 * FINAL SUMMARY:
 * > WRITE = correctness + safety
 * > READ  = speed + experience
 * > Ek hi system dono ke liye perfect nahi hota
 * > Isliye separation needed (CQRS)
*/

In [ ]:
/**
 * THE CORE IDEA OF CQRS:
 * ──────────────────────────────────────────
 * 
 *   Database ke andar sab operations do buckets mein aate hain:
 *   Bucket 1: Mutation (kuch badlo)  → COMMAND
 *   Bucket 2: Read (kuch padho)      → QUERY
 * 
 *   CQRS kehta hai: In dono ko SEPARATE rakho:
 *   - Separate models
 *   - Separatte services
 *   - Separate databases
 *
 * THE TWO OPERATIONS:
 *
 *   ┌──────────────────────────────────────────────────────────────────┐
 *   │  COMMAND (Mutation / Write)         QUERY (Read)                 │
 *   │  ──────────────────────────         ──────────────────────────   │
 *   │  → Place an order                   → Get my orders              │
 *   │  → Update inventory                 → Search products            │
 *   │  → Add to cart                      → View cart                  │
 *   │  → Submit a review                  → Read reviews               │
 *   │  → Process payment                  → Check order status         │
 *   │  → Cancel order                     → View recommendations       │
 *   │                                                                  │
 *   │  Returns: acknowledgement           Returns: data/results        │
 *   │  (success/failure, no data)         (the actual data you want)   │
 *   └──────────────────────────────────────────────────────────────────┘
 *
 * 
 * KEY PRINCIPLE (From CQS — Bertrand Meyer):
 * > "A function should either DO something (Command) or RETURN something (Query) — 
 *    but never both."
 * > CQRS kya karta hai: Is rule ko pura system level pe apply karta hai
 *
 * 
 * BEFORE CQRS (Mixed Approach):
 *   ┌──────────────────────────────────────────────────────────────────┐
 *   │  OrderService {                                                  │
 *   │      placeOrder(data)  ← WRITE + returns order details (mixed!)  │
 *   │      getOrders(userId) ← READ                                    │
 *   │      updateStatus(id)  ← WRITE                                   │
 *   │      searchOrders()    ← READ (heavy query)                      │
 *   │  }                                                               │
 *   |                                                                  |
 *   │ Problem:                                                         │
 *   │ - Sab ek hi service me mix                                       │
 *   │ - Same DB pe load                                                │
 *   │ - Scaling difficult                                              │
 *   └──────────────────────────────────────────────────────────────────┘
 *   
 * AFTER CQRS (Separated Approach):
 *   ┌─────────────────────────────┐    ┌──────────────────────────────┐
 *   │ OrderCommandService         │    │ OrderQueryService            │
 *   ├─────────────────────────────┤    ├──────────────────────────────┤
 *   │ - placeOrder(data)          │    │ - getOrderById(id)           │
 *   │ - cancelOrder(id)           │    │ - getOrdersByUser(userId)    │
 *   │ - updateOrderStatus(id, st) │    │ - searchOrders(filters)      │
 *   │                             │    │ - getOrderSummary(userId)    │
 *   ├─────────────────────────────┤    ├──────────────────────────────┤
 *   │ Uses: WRITE DB              │    │ Uses: READ DB                │
 *   │ (PostgreSQL / MySQL)        │    │ (MongoDB / Redis / Cache)    │
 *   └─────────────────────────────┘    └──────────────────────────────┘
 *   
 *
 * 50% LOAD SPLIT
 * > Example:
 *   - 50% operations → READ
 *   - 50% operations → WRITE
 * > Without CQRS:
 *   - Same DB pe dono load
 *   - Bottleneck create hota hai
 * > With CQRS:
 *   - WRITE load → sirf write DB handle kare
 *   - READ load  → read replicas / cache handle kare
 * 
 * SCALING BENEFITS:
 * > Write DB slow ho → usko optimize/upgrade karo
 * > Read slow ho → aur replicas add karo
 * > Dono independent scale karte hain
 *   - No resource waste
 *   - Better performance
 *   - Clean architecture
 */

In [ ]:
/**
 * Data Syncing with Event Sourcing:
 * ─────────────────────────────────
 * > Pehle user ki intention pehchani jaegi ki user krna kya chahta hai.
 * > Agar wo COMMAND krna chahta hai to command ka logic humein aata hi hai:
 *   - Validation
 *   - Business Logic
 *   - Command Execute (mutate data)
 *   - Data Persistence
 * > Agar wo QUERY krna chahta hai to uska v logic humein aata hai:
 *   - Validation
 *   - READ? 
 *   - Retrieve Data 
 * > Dono clearly separate paths hain: Command path aur Query path.
 * 
 * Problem: Sync Issue
 * > READ & WRITE k liye hm alag-alag databases use kar rhe hai. To kya humein kabhi
 *   data milega? No! Because databases are not in sync, jisse data missing lagta hai 
 *   aur humari consistency kharab ho jaati hai.
 * > Humne kuch WRITE kiya, ab hum usse READ krne jaenge to milega hi nhi kyuki READ
 *   and WRITE k liye alag alag database use kr rhe hai. So, how to keep the WRITE DB
 *   in sync with READ DB?
 * 
 * Solution: Event-Driven Architecture
 * > We need some kind of Event-Driven Architecture like Kafka, BullMQ, SQS, etc. to keep
 *   the databases in sync.
 * > Jab v WRITE DB m kuch hoga to wo ek event post krega, aur humara READ model will 
 *   read it and store it in READ DB. But can we store it in denormalized way? 
 *   - Yes! 
 *   - Write model normalized hota hai  (data redundancy avoid karne ke liye).
 *   - Read model denormalized hota hai (fast reads ke liye, joins avoid karo).
 *   - Same data, alag structure — alag purpose serve karte hain.
 * 
 * Example:
 *           +------------+                      +-----------+
 *   Step-1: | Write APIs |                      | Read APIs | Step-6 
 *           +------------+                      +-----------+      
 *                  |                                  |
 *           Step-2 | Command                   Step-7 | Query
 *                  V                                  V
 *           +------------+       Step-4         +-----------+
 *   Step-3: |  Write DB  |--------------------->|  Read DB  | Step-5
 *           +------------+        Event         +-----------+
 * 
 *  
 * 1. User clicks - "Place Order"
 * 2. WRITE:
 *    - OrderCommandHandler run hota hai
 *    - PostgreSQL me order save hota hai in normalized way
 * 3. Create Event:
 *    - Event publish hota hai:
 *      {
 *        type: "OrderPlaced",
 *        orderId: 123,
 *        userId: 45,
 *        items: [...],
 *        totalAmount: 999
 *      }
 * 4. Queue:
 *    - Event Kafka/SQS me chala jaata hai
 * 5. READ:
 *    - Event Handler event consume krta hai
 *    - READ DB (MongoDB/Redis) me data ko denormalized kr k store krte hai, means:
 *      a. Multiple tables join krne ki zarurat nhi hai.
 *      b. Sab data ek document m ready.
 *         {
 *           orderId: 123,
 *           userName: "Mohammad",
 *           items: [...],
 *           totalAmount: 999,
 *           status: "PLACED",
 *           createdAt: "..."
 *         }
 *    - Benefit:
 *      a. Super fast reads
 *      b. No heavy queries
 *      c. Flexibility
*/

In [ ]:
/**
 * CQRS Architecture:
 * 
 *   ┌─────────────────────────────────────────────────────────────────────┐
 *   │                         CLIENT                                      │
 *   │            (Browser/Mobile App/External Service)                    │
 *   └─────────────────────────────────────────────────────────────────────┘
 *                    │                          │
 *             WRITE request                READ request
 *             POST /orders                 GET /orders/123
 *             PUT /inventory               GET /products?search=...
 *                    │                          │
 *                    ▼                          ▼
 *   ┌────────────────────────┐    ┌─────────────────────────────────────┐
 *   │   COMMAND HANDLER      │    │         QUERY HANDLER               │
 *   │                        │    │                                     │
 *   │  PlaceOrderCommand     │    │  GetOrderQuery                      │
 *   │  UpdateInventoryCommand│    │  SearchProductsQuery                │
 *   │  CancelOrderCommand    │    │  GetOrderHistoryQuery               │
 *   │                        │    │                                     │
 *   │  → Validates input     │    │  → No validation needed             │
 *   │  → Runs business logic │    │  → Just fetch and return data       │
 *   │  → Applies domain rules│    │  → Optimized for speed              │
 *   └────────────────────────┘    └─────────────────────────────────────┘
 *                    │                          │
 *                    ▼                          ▼
 *   ┌────────────────────────┐    ┌─────────────────────────────────────┐
 *   │    WRITE DATABASE      │    │         READ DATABASE               │
 *   │────────────────────────│    │─────────────────────────────────────│
 *   │   PostgreSQL/MySQL     │    │    MongoDB/DynamoDB/Cassandra       │
 *   │                        │    │  + Redis (cache)                    │
 *   │  Normalized schema     │    │  + Elasticsearch (search)           │
 *   │  ACID transactions     │    │                                     │
 *   │  Strong consistency    │    │  Denormalized, flat documents       │
 *   │  Source of truth       │    │  Pre-aggregated data                │
 *   │                        │    │  Fast reads, no joins               │
 *   └────────────────────────┘    └─────────────────────────────────────┘
 *                    │                          ▲
 *                    │                          │
 *                    │      EVENT / MESSAGE     │
 *                    └──────────────────────────┘
 *                       [Kafka / SQS / BullMQ]
 *             (OrderPlaced event → update read DB asynchronously)
 *
 * SYNC MECHANISM (How read DB stays up to date):
 *
 *   ┌──────────────────────────────────────────────────────────────────┐
 *   │  1. Command Handler writes to WRITE DB                           │
 *   │  2. An EVENT is published: "OrderPlaced"/"InventoryUpdated"      │
 *   │  3. Event goes to MESSAGE QUEUE (Kafka/RabbitMQ/SQS)             │
 *   │  4. EVENT HANDLER consumes the event                             │
 *   │  5. Event Handler UPDATES the READ DB                            │
 *   │  6. Future queries now see the updated data                      │
 *   │                                                                  │
 *   │  Gap between write and read DB being in sync: a few ms to secs   │
 *   │  This is called EVENTUAL CONSISTENCY.                            │
 *   └──────────────────────────────────────────────────────────────────┘
 */

In [ ]:
/**
 * READ MODEL VS WRITE MODEL — Schema Difference:
 * > Write model normalized hota hai (data redundancy avoid karne ke liye).
 * > Read model denormalized hota hai (fast reads ke liye, joins avoid karo).
 * > Same data, alag structure — alag purpose serve karte hain.
 *
 * WRITE MODEL (Normalized — Source of Truth):
 *
 *   orders table:
 *   ┌────────────┬──────────┬────────────┬───────────┬────────────────┐
 *   │ id         │ user_id  │ address_id │ status    │ created_at     │
 *   ├────────────┼──────────┼────────────┼───────────┼────────────────┤
 *   │ ord_001    │ usr_42   │ addr_7     │ SHIPPED   │ 2024-01-15     │
 *   └────────────┴──────────┴────────────┴───────────┴────────────────┘
 *
 *   order_items table:
 *   ┌────────────┬────────────┬────────────┬──────────┐
 *   │ id         │ order_id   │ product_id │ quantity │
 *   ├────────────┼────────────┼────────────┼──────────┤
 *   │ item_001   │ ord_001    │ prod_55    │ 2        │
 *   │ item_002   │ ord_001    │ prod_89    │ 1        │
 *   └────────────┴────────────┴────────────┴──────────┘
 *
 * > products table, users table, addresses table ... all separate
 * > To display ORDER DETAILS page → 5 JOINs needed → SLOW at scale
 *
 * 
 * READ MODEL (Denormalized — Optimized for Display):
 *
 *   orders_read collection (MongoDB):
 *   {
 *     "orderId": "ord_001",
 *     "status": "SHIPPED",
 *     "createdAt": "2024-01-15",
 *     "user": {
 *       "id": "usr_42",
 *       "name": "Rahul Sharma",
 *       "email": "rahul@example.com"
 *     },
 *     "deliveryAddress": {
 *       "street": "42 MG Road",
 *       "city": "Mumbai",
 *       "pincode": "400001"
 *     },
 *     "items": [
 *       {
 *         "productId": "prod_55",
 *         "productName": "iPhone 15",
 *         "productImage": "iphone15.jpg",
 *         "quantity": 2,
 *         "price": 79999,
 *         "subtotal": 159998
 *       },
 *       {
 *         "productId": "prod_89",
 *         "productName": "AirPods Pro",
 *         "quantity": 1,
 *         "price": 24999,
 *         "subtotal": 24999
 *       }
 *     ],
 *     "totalAmount": 184997,
 *     "trackingInfo": {
 *       "carrier": "BlueDart",
 *       "trackingId": "BD123456789",
 *       "estimatedDelivery": "2024-01-17"
 *     }
 *   }
 *
 * > To display ORDER DETAILS page → ONE document fetch → SUPER FAST!
 *
 * 
 * KEY INSIGHT:
 * > Write model se ek baar write karo → event publish karo.
 * > Event handler ye flat document banake read DB mein store kare.
 * > Ab har read sirf ek document fetch karta hai — no joins, no complexity.
 *
 * 
 * MULTIPLE READ MODELS FOR DIFFERENT VIEWS:
 *   ┌──────────────────────────────────────────────────────────────────┐
 *   │  "My Orders" page     → orders_by_user collection                │
 *   │  "Order Details" page → order_detail collection                  │
 *   │  "Admin Dashboard"    → orders_admin_view collection             │
 *   │  "Search"             → Elasticsearch index                      │
 *   │  "Recommendations"    → Redis sorted sets                        │
 *   └──────────────────────────────────────────────────────────────────┘
 *
 * > Ek hi WRITE model se multiple READ models ban sakte hain!
 * > Har view ke liye perfectly optimized read model — blazing fast reads.
 */

In [ ]:
/**
 * SYNCHRONIZATION — How Read DB Stays in Sync:
 * 
 * Read DB Ko Update Kaise Karte Hain — Event-Driven Sync
 * ──────────────────────────────────────────────────────────
 * 
 *   Write DB mein kuch badla 
 *   → event publish hota hai 
 *   → message queue mein jaata hai
 *   → event consumer read DB update karta hai.
 * 
 *   Ye process asynchronous hai — thodi der lag sakti hai (eventual consistency).
 *   Lekin ye delay acceptable hai — "Order placed!" dikha do, detail 2 seconds mein aayegi.
 *
 * SYNC FLOW (Step by Step):
 *
 *   ┌──────────────────────────────────────────────────────────────────────┐
 *   │  STEP 1: User places an order                                        │
 *   │          POST /api/orders  { productId, quantity, address }          │
 *   └──────────────────────────────────────────────────────────────────────┘
 *                    │
 *                    ▼
 *   ┌──────────────────────────────────────────────────────────────────────┐
 *   │  STEP 2: Command Handler processes the order                         │
 *   │          → Validates stock, payment, address                         │
 *   │          → INSERT into orders table (WRITE DB)                       │
 *   │          → UPDATE inventory SET stock = stock - 1 (WRITE DB)         │
 *   │          → Returns: { orderId: "ord_001", status: "CONFIRMED" }      │
 *   └──────────────────────────────────────────────────────────────────────┘
 *                    │
 *                    ▼
 *   ┌──────────────────────────────────────────────────────────────────────┐
 *   │  STEP 3: Event published to Message Queue (Kafka)                    │
 *   │                                                                      │
 *   │  Topic: "order-events"                                               │
 *   │  Event: {                                                            │
 *   │    type: "ORDER_PLACED",                                             │
 *   │    orderId: "ord_001",                                               │
 *   │    userId: "usr_42",                                                 │
 *   │    productIds: ["prod_55", "prod_89"],                               │
 *   │    timestamp: "2024-01-15T10:30:00Z"                                 │
 *   │  }                                                                   │
 *   └──────────────────────────────────────────────────────────────────────┘
 *                    │
 *                    ▼
 *   ┌──────────────────────────────────────────────────────────────────────┐
 *   │  STEP 4: Multiple Event Consumers process the event INDEPENDENTLY    │
 *   │                                                                      │
 *   │  Consumer 1: OrderReadModelUpdater                                   │
 *   │    → Fetches full order data from WRITE DB                           │
 *   │    → Builds denormalized document                                    │
 *   │    → Upserts into orders_read collection (MongoDB)                   │
 *   │                                                                      │
 *   │  Consumer 2: SearchIndexUpdater                                      │
 *   │    → Updates Elasticsearch index (order searchable now)              │
 *   │                                                                      │
 *   │  Consumer 3: NotificationService                                     │
 *   │    → Sends "Order confirmed!" email/SMS to user                      │
 *   │                                                                      │
 *   │  Consumer 4: AnalyticsService                                        │
 *   │    → Updates sales dashboard metrics                                 │
 *   └──────────────────────────────────────────────────────────────────────┘
 *                    │
 *                    ▼
 *   ┌──────────────────────────────────────────────────────────────────────┐
 *   │  STEP 5: User queries their order (a few ms-seconds later)           │
 *   │          GET /api/orders/ord_001                                     │
 *   │          → Query Handler reads from READ DB (MongoDB)                │
 *   │          → Returns the pre-built denormalized document               │
 *   │          → Response in <10ms ✅                                      │
 *   └──────────────────────────────────────────────────────────────────────┘
 *
 * EVENTUAL CONSISTENCY EXPLAINED:
 *
 *   ┌──────────────────────────────────────────────────────────────────┐
 *   │  T+0ms:   Order placed → Write DB updated                        │
 *   │  T+50ms:  Event published to Kafka                               │
 *   │  T+100ms: Event Consumer updates Read DB                         │
 *   │  T+200ms: Read DB is consistent with Write DB                    │
 *   │                                                                  │
 *   │  During T+0 to T+200ms:                                          │
 *   │    User sees old order list (eventually consistent)              │
 *   │    This is ACCEPTABLE — user just placed order, they know!       │
 *   │    We show optimistic UI: "Order placed! Details loading..."     │
 *   └──────────────────────────────────────────────────────────────────┘
 */

In [ ]:
/**
 * CQRS + EVENT SOURCING
 * 
 * 
 * Event Sourcing — CQRS Ka Powerful Partner
 * ──────────────────────────────────────────
 * 
 *   CQRS akela bhi use ho sakta hai — aur Event Sourcing ke saath bhi.
 *   Event Sourcing = data ka current state store karne ki jagah
 *                    har event store karo jo us state tak pahuncha.
 *   Jaise bank account — balance directly store nahi karte,
 *   sab transactions store karte hain, balance calculate karte hain.
 *
 * TRADITIONAL STORAGE vs EVENT SOURCING:
 *
 *   TRADITIONAL (Stores current state):
 *   orders table: { id: "ord_001", status: "DELIVERED", updatedAt: ... }
 *   (Previous states LOST — we don't know it was PENDING → CONFIRMED → SHIPPED → DELIVERED)
 *
 *   EVENT SOURCING (Stores all events):
 *   order_events:
 *   ┌──────────┬────────────────────┬─────────────────────────────────────┐
 *   │ event_id │ type               │ data                                │
 *   ├──────────┼────────────────────┼─────────────────────────────────────┤
 *   │ 1        │ ORDER_PLACED       │ { orderId, userId, items, amount }  │
 *   │ 2        │ PAYMENT_CONFIRMED  │ { orderId, transactionId }          │
 *   │ 3        │ ORDER_SHIPPED      │ { orderId, trackingId, carrier }    │
 *   │ 4        │ ORDER_DELIVERED    │ { orderId, deliveredAt }            │
 *   └──────────┴────────────────────┴─────────────────────────────────────┘
 *
 *   Current state = replay all events from beginning.
 *   Complete audit trail — kuch bhi reconstruct kar sakte hain.
 *
 * HOW CQRS + EVENT SOURCING WORK TOGETHER:
 *
 *   ┌─────────────────────────────────────────────────────────────────────┐
 *   │  Command → Validates → Produces EVENT → Stores in EVENT STORE       │
 *   │                                │                                    │
 *   │                                ▼                                    │
 *   │                         Event published                             │
 *   │                                │                                    │
 *   │              ┌─────────────────┼──────────────────┐                 │
 *   │              ▼                 ▼                  ▼                 │
 *   │       Read Model 1      Read Model 2       Read Model 3             │
 *   │       (MongoDB)         (Elasticsearch)    (Redis Cache)            │
 *   │       Orders by User    Search Index       Dashboard Stats          │
 *   └─────────────────────────────────────────────────────────────────────┘
 *
 * BENEFITS OF EVENT SOURCING:
 *   ✅ Complete audit trail (compliance ke liye perfect)
 *   ✅ Time travel — "what was the state at 2pm yesterday?"
 *   ✅ Replay events to rebuild any read model
 *   ✅ Bug fix karo, purane events dobara replay karo → fixed read model
 *   ✅ New feature? Replay all past events → instant historical data
 */

In [ ]:
/**
 * REAL-WORLD EXAMPLE — Amazon E-Commerce CQRS Flow
 */
 
/**
 * Amazon Ka Order Flow — CQRS Ke Saath
 * ──────────────────────────────────────
 * 
 *   Ab dekhte hain ek real Amazon-like scenario end to end —
 *   customer product browse karta hai, cart mein daalta hai,
 *   order place karta hai, aur track karta hai.
 *
 * SCENARIO: Rahul buys an iPhone 15 on Amazon
 *
 * ─── PHASE 1: Product Browse (QUERY) ─────────────────────────────────────
 *
 *   Rahul → GET /api/products?search=iphone&category=mobiles
 *
 *   ┌──────────────────────────────────────────────────────────────────┐
 *   │  QUERY PATH                                                      │
 *   │                                                                  │
 *   │  QueryHandler → Elasticsearch products index                     │
 *   │    → Returns: pre-built product cards with                       │
 *   │               name, image, price, rating, stock                  │
 *   │    → Response time: 5-15ms                                       │
 *   │    → No joins, no complex queries                                │
 *   └──────────────────────────────────────────────────────────────────┘
 *
 * ─── PHASE 2: Add to Cart (COMMAND) ──────────────────────────────────────
 *
 *   Rahul → POST /api/cart/items { productId: "prod_iphone15", qty: 1 }
 *
 *   ┌──────────────────────────────────────────────────────────────────┐
 *   │  COMMAND PATH                                                    │
 *   │                                                                  │
 *   │  CommandHandler:                                                 │
 *   │    → Validate: is product available?                             │
 *   │    → Write: INSERT cart_item (WRITE DB - PostgreSQL)             │
 *   │    → Publish: CART_ITEM_ADDED event                              │
 *   │    → Return: { success: true }                                   │
 *   │                                                                  │
 *   │  Event Consumer (async):                                         │
 *   │    → Update cart_read document in MongoDB                        │
 *   │    → Update cart count in Redis cache                            │
 *   └──────────────────────────────────────────────────────────────────┘
 *
 * ─── PHASE 3: View Cart (QUERY) ───────────────────────────────────────────
 *
 *   Rahul → GET /api/cart
 *
 *   ┌──────────────────────────────────────────────────────────────────┐
 *   │  QUERY PATH                                                      │
 *   │                                                                  │
 *   │  QueryHandler → MongoDB cart_read collection                     │
 *   │    → Returns: pre-built cart document with                       │
 *   │               product details, prices, totals                    │
 *   │    → Response time: 3-8ms (Redis cache hit!)                     │
 *   └──────────────────────────────────────────────────────────────────┘
 *
 * ─── PHASE 4: Place Order (COMMAND) ──────────────────────────────────────
 *
 *   Rahul → POST /api/orders { cartId, addressId, paymentMethod }
 *
 *   ┌──────────────────────────────────────────────────────────────────┐
 *   │  COMMAND PATH (most complex — validation heavy)                  │
 *   │                                                                  │
 *   │  CommandHandler:                                                 │
 *   │    → Validate stock (check inventory)                            │
 *   │    → Validate payment (call payment gateway)                     │
 *   │    → BEGIN TRANSACTION (ACID)                                    │
 *   │       → INSERT orders table                                      │
 *   │       → INSERT order_items table                                 │
 *   │       → UPDATE inventory (stock - 1)                             │
 *   │       → UPDATE cart (mark as checked out)                        │
 *   │    → COMMIT TRANSACTION                                          │
 *   │    → Return: { orderId: "ord_001", status: "CONFIRMED" }         │
 *   │                                                                  │
 *   │  Events published (async):                                       │
 *   │    → ORDER_PLACED event → Kafka                                  │
 *   │                                                                  │
 *   │  Multiple Consumers:                                             │
 *   │    → Read Model Updater → builds order_read document (MongoDB)   │
 *   │    → Notification Service → sends confirmation email/SMS         │
 *   │    → Analytics Service → updates sales metrics                   │
 *   │    → Warehouse Service → triggers shipment preparation           │
 *   │    → Search Index → marks item stock reduced                     │
 *   └──────────────────────────────────────────────────────────────────┘
 *
 * ─── PHASE 5: Track Order (QUERY) ─────────────────────────────────────────
 *
 *   Rahul → GET /api/orders/ord_001
 *
 *   ┌──────────────────────────────────────────────────────────────────┐
 *   │  QUERY PATH                                                      │
 *   │                                                                  │
 *   │  QueryHandler → MongoDB order_read collection                    │
 *   │    → Returns: complete pre-built order document                  │
 *   │               (user, items, address, tracking) — NO JOINS        │
 *   │    → Response time: 5-10ms                                       │
 *   └──────────────────────────────────────────────────────────────────┘
 *
 * RESULT: Millions of users can browse, order, track simultaneously
 *         because READ and WRITE paths are completely independent!
 */

In [ ]:
/**
 *  WHEN TO USE CQRS
 * ─────────────────────
 * → CQRS tab use karo jab normal system struggle karne lage
 * → Har project me use karna zaroori nahi hai
 *
 * USE CQRS WHEN:
 *
 * 1. High traffic / large scale system
 *    → Bahut zyada users (10M+), heavy load
 *    → Database slow ho raha hai ya bottleneck ban gaya
 *    → Example: big apps like Amazon, Flipkart
 *
 * 2. Read aur Write needs alag-alag hain
 *    → Write: strong validation + correct data (ACID)
 *    → Read: super fast response + complex queries
 *    → Dono ko alag handle karna better hota hai
 *
 * 3. Complex business logic (heavy rules)
 *    → Write side me bahut saare rules, validations
 *    → Read side simple hota hai (sirf data dikhana)
 *    → CQRS se clean separation milta hai
 *
 * 4. Same data ke multiple views chahiye
 *    → Ek hi data ko alag-alag format me dikhana hai
 *    → Example:
 *         - Customer view
 *         - Admin dashboard
 *         - Warehouse system
 *    → CQRS me multiple read models banana easy hota hai
 *
 * 5. Audit trail / history track karna zaroori hai
 *    → Kisne kya change kiya, kab kiya → sab track karna hai
 *    → Event sourcing + CQRS best combo hai
 *    → Example: banking, healthcare systems
 *
 * 6. Microservices architecture use kar rahe ho
 *    → Har service apna data khud manage karti hai
 *    → Services events ke through baat karti hain
 *    → CQRS naturally fit ho jata hai
 * 
 * FINAL ADVICE:
 * → Jab system scale ya complexity handle nahi kar pa raha ho → CQRS use karo
 * → Warna simple architecture hi best hai
 */

In [ ]:
/**
 * WHEN NOT TO USE CQRS
 * ──────────────────────
 * → CQRS powerful hai, but har jagah use karna smart nahi hota
 * → Agar zarurat nahi hai, to yeh system ko unnecessarily complex bana deta hai
 *
 * DO NOT USE CQRS IN THESE CASES:
 * 
 * 1. Small user base (< 1 lakh users)
 *    → Ek single database hi kaafi hota hai
 *    → CQRS lagane ka koi real benefit nahi
 *    → Simple CRUD + indexes = best choice
 *
 * 2. Simple CRUD applications
 *    → Jaise admin panel, blog, internal tools
 *    → Read aur write dono simple hote hain
 *    → Split karne ki zarurat hi nahi
 *
 * 3. Small team (no distributed systems experience)
 *    → CQRS samajhna tough hota hai
 *    → Aapko yeh sab aana chahiye:
 *         - Message queues (Kafka, RabbitMQ)
 *         - Eventual consistency
 *         - Multiple databases
 *         - Distributed systems concepts
 *    → Galat implementation = bade bugs
 *
 * 4. When strong consistency is required
 *    → Jaise banking system
 *    → Write ke turant baad read me same data dikhna chahiye
 *    → CQRS me delay ho sakta hai (eventual consistency)
 *
 * 5. Early stage startup
 *    → Pehle product build karo, idea validate karo
 *    → Users aane do, scale baad me handle karo
 *    → Over-engineering avoid karo
 *
 * RULE OF THUMB:
 *   ┌──────────────────────────────────────────────────────────────────┐
 *   │  1 lakh users  → Simple CRUD, single DB                          │
 *   │  10 lakh users → Read replicas, caching, indexes                 │
 *   │  1 crore users → Consider CQRS for specific heavy features       │
 *   │  10 crore+     → CQRS is almost certainly needed                 │
 *   └──────────────────────────────────────────────────────────────────┘
 * 
 * FINAL ADVICE:
 * → Jab tak real scaling problem na aaye, CQRS mat use karo
 * → Simple rakho, phir gradually upgrade karo
*/

In [ ]:
/**
 * CQRS BENEFITS AND TRADE-OFFS
 * ──────────────────────────────────────────────────
 * → Write (data save/update) aur Read (data fetch) alag systems handle karte hain.
 * → Isse performance aur scaling improve hoti hai.
 * → Lekin dhyaan rahe — benefits ke saath trade-offs bhi aate hain.
 * 
 * ┌──────────────────────────────────────┬────────────────────────────────────────────┐
 * │ BENEFITS (Faayde)                    │ TRADE-OFFS (Nuksaan / Challenges)          │
 * ├──────────────────────────────────────┼────────────────────────────────────────────┤
 * │ Read aur Write alag scale ho sakte   │ Data thoda delay se update dikhega         │
 * │ hain (better performance)            │ (eventual consistency)                     │
 * ├──────────────────────────────────────┼────────────────────────────────────────────┤
 * │ Read fast hota hai (no JOINs,        │ Same data multiple jagah store hota hai    │
 * │ optimized data)                      │ (duplication)                              │
 * ├──────────────────────────────────────┼────────────────────────────────────────────┤
 * │ Write side strong hota hai           │ System complex ho jata hai (multiple DB,   │
 * │ (validation, ACID rules)             │ queues, services)                          │
 * ├──────────────────────────────────────┼────────────────────────────────────────────┤
 * │ Alag-alag read views bana sakte      │ Zyada code likhna padta hai (handlers,     │
 * │ hain (custom dashboards etc.)        │ consumers, models)                         │
 * ├──────────────────────────────────────┼────────────────────────────────────────────┤
 * │ Agar read system fail ho jaye,       │ Debugging mushkil ho sakti hai             │
 * │ write fir bhi kaam karega            │ (events chain track karna tough)           │
 * ├──────────────────────────────────────┼────────────────────────────────────────────┤
 * │ Code clean aur easy to understand    │ Team ko distributed systems ka knowledge   │
 * │ (clear separation)                   │ chahiye                                    │
 * ├──────────────────────────────────────┼────────────────────────────────────────────┤
 * │ Naya read model easily add kar       │ Read data kabhi-kabhi outdated ho sakta    │
 * │ sakte ho bina write change kiye      │ hai (temporary mismatch)                   │
 * ├──────────────────────────────────────┼────────────────────────────────────────────┤
 * │ High performance (millions of        │ Cost badh jata hai (extra servers, DBs)    │
 * │ users handle kar sakta hai)          │                                            │
 * └──────────────────────────────────────┴────────────────────────────────────────────┘
 *
 *
 * TRADITIONAL vs CQRS COMPARISON:
 * → Traditional: ek hi system read + write dono handle karta hai
 * → CQRS: read aur write alag-alag systems handle karte hain
 *
 * ┌──────────────────────────────┬──────────────────────────────┬──────────────────────────────┐
 * │ Aspect                       │ Traditional CRUD             │ CQRS                         │
 * ├──────────────────────────────┼──────────────────────────────┼──────────────────────────────┤
 * │ Complexity (Kitna tough)     │ Simple                       │ Complex (samajhna mushkil)   │
 * ├──────────────────────────────┼──────────────────────────────┼──────────────────────────────┤
 * │ Infrastructure               │ Single database              │ Multiple DB + Queue system   │
 * │ (System setup)               │                              │ (Kafka, SQS etc.)            │
 * ├──────────────────────────────┼──────────────────────────────┼──────────────────────────────┤
 * │ Read Performance             │ Good (indexes use karke)     │ Very fast (optimized reads)  │
 * │ (Data fetch speed)           │                              │                              │
 * ├──────────────────────────────┼──────────────────────────────┼──────────────────────────────┤
 * │ Write Performance            │ Good                         │ Good (separate write system) │
 * │ (Data save/update)           │                              │                              │
 * ├──────────────────────────────┼──────────────────────────────┼──────────────────────────────┤
 * │ Scalability                  │ Limited (single DB limit)    │ Easily scalable (read/write  │
 * │ (Growth handle karna)        │                              │ alag scale kar sakte hain)   │
 * ├──────────────────────────────┼──────────────────────────────┼──────────────────────────────┤
 * │ Consistency                  │ Strong (instant update)      │ Slight delay (eventual       │
 * │ (Data freshness)             │                              │ consistency)                 │
 * ├──────────────────────────────┼──────────────────────────────┼──────────────────────────────┤
 * │ Suitable for                 │ Small apps (<1M users)       │ Large apps (10M+ users)      │
 * │ (Use case)                   │                              │                              │
 * ├──────────────────────────────┼──────────────────────────────┼──────────────────────────────┤
 * │ Learning Curve               │ Easy                         │ Hard (concepts zyada)        │
 * ├──────────────────────────────┼──────────────────────────────┼──────────────────────────────┤
 * │ Cost                         │ Low                          │ High (infra + dev effort)    │
 * └──────────────────────────────┴──────────────────────────────┴──────────────────────────────┘
 *
 * FINAL ADVICE:
 * → Agar tumhara app simple hai               → Traditional best
 * → Agar tumhe scale chahiye (millions users) → CQRS consider karo
 */

In [ ]:
/**
 * QUICK REFERENCE:
 * 
 * 1. COMMAND side:  POST/PUT/PATCH/DELETE routes
 *                   → Validates → Writes to WRITE DB → Publishes Event
 *                   → Returns: acknowledgement only (no data)
 *
 * 2. QUERY side:    GET routes
 *                   → No validation → Reads from READ DB
 *                   → Returns: data (fast, no joins)
 *
 * 3. SYNC:          WRITE DB → Event → Message Queue (Kafka)
 *                            → Event Consumer → READ DB updated
 *
 * 4. CONSISTENCY:   Eventual (few ms to seconds lag)
 *
 * 5. SCALE:         Write DB: vertical scale (more CPU/RAM)
 *                   Read DB:  horizontal scale (more replicas, more nodes)
 */
 